In [1]:
#!/usr/bin/env python3
"""
ERROR PATTERN ANALYSIS
======================
Analyzes which country characteristics predict each model's errors.
This uses EXISTING predictions - NO NEW API CALLS NEEDED.

Tests:
1. Correlation between errors and country features
2. Regression: Which features predict errors?
3. Geographic/regional error patterns
4. GDP/development-related patterns
"""

# ================================================================
# Fix matplotlib compatibility
# ================================================================
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")

print("="*80)
print("ERROR PATTERN ANALYSIS")
print("="*80)
print("Analyzing what predicts each model's prediction errors")
print("Uses existing predictions - NO new API calls needed")

# ================================================================
# 1. Load Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

# Load predictions
df = pd.read_csv("predictions_all_stages_long.csv")
model_cols = [col for col in df.columns if col.startswith('pred_')]

# Add continent mapping
continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}
df['continent'] = df['countrynew'].map(continent_mapping)

# Load ground truth
try:
    gt_df = pd.read_csv("data_final.csv")
    feature_cols = ['countrynew', 'mean_age', 'mean_edu', 'mean_religion',
                   'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth',
                   'hdi_2021', 'mean_temp_2010_2019', 'mean_own_willingness',
                   'mean_other_willingness']
    available_cols = [col for col in feature_cols if col in gt_df.columns]
    gt_df = gt_df[available_cols]
    df = df.merge(gt_df, on='countrynew', how='left')
    df['ground_truth_pi'] = df['mean_other_willingness'] * 100
    
    print(f"✓ Loaded {len(df)} predictions")
    print(f"✓ Countries: {df['countrynew'].nunique()}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    exit()

# Focus on Stage 8 (maximum information)
stage8_df = df[df['stage'] == 8].copy()
stage8_df = stage8_df.dropna(subset=['ground_truth_pi'])

print(f"✓ Analyzing Stage 8: {len(stage8_df)} countries")

# Calculate errors for each model
models_to_analyze = {
    'Llama': 'pred_llama',
    'GPT': 'pred_gpt',
    'Claude': 'pred_claude',
    'Gemini': 'pred_gemini'
}

for model_name, col in models_to_analyze.items():
    if col in stage8_df.columns:
        stage8_df[f'{col}_error'] = stage8_df[col] - stage8_df['ground_truth_pi']
        stage8_df[f'{col}_abs_error'] = np.abs(stage8_df[f'{col}_error'])
        print(f"✓ Calculated errors for {model_name}")

# ================================================================
# 2. Correlation Analysis
# ================================================================

print("\n" + "="*80)
print("2. CORRELATION ANALYSIS: Features vs Errors")
print("="*80)

# Define features to test
features_to_test = {
    'GDP per capita': 'gdp_capita_2021',
    'Education (years)': 'mean_edu',
    'Religious importance': 'mean_religion',
    'HDI': 'hdi_2021',
    'Top 1% income': 'top1pct_income',
    'Top 1% wealth': 'top1pct_wealth',
    'Mean age': 'mean_age',
    'Temperature': 'mean_temp_2010_2019'
}

correlation_results = []

for model_name, col in models_to_analyze.items():
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    print(f"\n{model_name}:")
    print("-" * 60)
    
    for feat_name, feat_col in features_to_test.items():
        if feat_col in stage8_df.columns:
            # Remove NaNs
            valid_data = stage8_df[[feat_col, error_col]].dropna()
            
            if len(valid_data) > 10:
                r, p = pearsonr(valid_data[feat_col], valid_data[error_col])
                
                significance = ""
                if p < 0.001:
                    significance = "***"
                elif p < 0.01:
                    significance = "**"
                elif p < 0.05:
                    significance = "*"
                
                print(f"   {feat_name:25s}: r = {r:6.3f}, p = {p:.4f} {significance}")
                
                correlation_results.append({
                    'model': model_name,
                    'feature': feat_name,
                    'correlation': r,
                    'p_value': p,
                    'significant': p < 0.05
                })

# Save correlation results
corr_df = pd.DataFrame(correlation_results)
corr_df.to_csv('error_correlations.csv', index=False)
print("\n✓ Saved error_correlations.csv")

# ================================================================
# 3. Regression Analysis
# ================================================================

print("\n" + "="*80)
print("3. REGRESSION ANALYSIS: Predicting Errors")
print("="*80)

# Prepare feature matrix
feature_cols = [col for col in ['gdp_capita_2021', 'mean_edu', 'mean_religion', 
                                'hdi_2021', 'top1pct_income', 'mean_age']
               if col in stage8_df.columns]

X = stage8_df[feature_cols].dropna()
regression_results = {}

for model_name, col in models_to_analyze.items():
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    # Get corresponding y values
    y = stage8_df.loc[X.index, error_col]
    
    # Fit regression
    reg = LinearRegression()
    reg.fit(X, y)
    
    r2 = reg.score(X, y)
    
    print(f"\n{model_name}:")
    print(f"   R² = {r2:.3f}")
    print(f"   Feature coefficients:")
    
    for feat, coef in zip(feature_cols, reg.coef_):
        print(f"      {feat:20s}: β = {coef:7.4f}")
    
    regression_results[model_name] = {
        'r2': r2,
        'coefficients': dict(zip(feature_cols, reg.coef_)),
        'intercept': reg.intercept_
    }

# ================================================================
# 4. Geographic Patterns
# ================================================================

print("\n" + "="*80)
print("4. GEOGRAPHIC ERROR PATTERNS")
print("="*80)

for model_name, col in models_to_analyze.items():
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    print(f"\n{model_name} - MAE by Continent:")
    print("-" * 60)
    
    continent_errors = stage8_df.groupby('continent')[error_col].agg(['mean', 'std', 'count'])
    continent_errors = continent_errors.sort_values('mean')
    
    for continent in continent_errors.index:
        mean_err = continent_errors.loc[continent, 'mean']
        std_err = continent_errors.loc[continent, 'std']
        n = continent_errors.loc[continent, 'count']
        print(f"   {continent:20s}: {mean_err:5.2f} ± {std_err:4.2f}pp (n={n:.0f})")

# ================================================================
# 5. Development Level Patterns
# ================================================================

print("\n" + "="*80)
print("5. ERRORS BY DEVELOPMENT LEVEL")
print("="*80)

# Create GDP quartiles
stage8_df['gdp_quartile'] = pd.qcut(stage8_df['gdp_capita_2021'], 
                                     q=4, labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'],
                                     duplicates='drop')

for model_name, col in models_to_analyze.items():
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    print(f"\n{model_name} - MAE by GDP Quartile:")
    print("-" * 60)
    
    for quartile in ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']:
        subset = stage8_df[stage8_df['gdp_quartile'] == quartile]
        if len(subset) > 0:
            mean_err = subset[error_col].mean()
            print(f"   {quartile:15s}: {mean_err:5.2f}pp (n={len(subset)})")

# ================================================================
# 6. Best and Worst Predictions
# ================================================================

print("\n" + "="*80)
print("6. BEST AND WORST PREDICTIONS")
print("="*80)

for model_name, col in models_to_analyze.items():
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    print(f"\n{model_name}:")
    print("-" * 60)
    
    # Best 5
    best = stage8_df.nsmallest(5, error_col)
    print("   BEST (smallest errors):")
    for _, row in best.iterrows():
        print(f"      {row['countrynew']:20s}: Error = {row[error_col]:4.1f}pp " +
              f"(Pred: {row[col]:5.1f}, Truth: {row['ground_truth_pi']:5.1f})")
    
    # Worst 5
    worst = stage8_df.nlargest(5, error_col)
    print("\n   WORST (largest errors):")
    for _, row in worst.iterrows():
        print(f"      {row['countrynew']:20s}: Error = {row[error_col]:4.1f}pp " +
              f"(Pred: {row[col]:5.1f}, Truth: {row['ground_truth_pi']:5.1f})")

# ================================================================
# 7. Visualizations
# ================================================================

print("\n" + "="*80)
print("7. CREATING VISUALIZATIONS")
print("="*80)

# Figure 1: Correlation heatmap
fig = plt.figure()
fig.set_size_inches(12, 8)
fig.set_dpi(300)
matplotlib.rcParams.update({})
ax = fig.add_subplot(111)

# Create correlation matrix
corr_matrix = corr_df.pivot(index='feature', columns='model', values='correlation')
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
           vmin=-0.5, vmax=0.5, ax=ax, cbar_kws={'label': 'Correlation (r)'})
ax.set_title('Feature Correlations with Model Errors\n(Positive = Feature predicts larger errors)',
            fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Country Feature', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('error_feature_correlations.png', dpi=300, bbox_inches='tight')
plt.savefig('error_feature_correlations.pdf', bbox_inches='tight')
print("   ✓ Saved error_feature_correlations.png")
print("   ✓ Saved error_feature_correlations.pdf")
plt.close()

# Figure 2: Scatter plots for key relationships
fig = plt.figure()
fig.set_size_inches(14, 10)
fig.set_dpi(300)
matplotlib.rcParams.update({})
axes = []
for i in range(4):
    ax = fig.add_subplot(2, 2, i + 1)
    axes.append(ax)

key_features = ['gdp_capita_2021', 'mean_religion', 'hdi_2021', 'mean_edu']
feature_names = ['GDP per Capita', 'Religious Importance', 'HDI', 'Education (years)']

for idx, (feat_col, feat_name) in enumerate(zip(key_features, feature_names)):
    if feat_col not in stage8_df.columns:
        continue
    
    ax = axes[idx]
    
    for model_name, col in models_to_analyze.items():
        error_col = f'{col}_abs_error'
        if error_col in stage8_df.columns:
            valid_data = stage8_df[[feat_col, error_col]].dropna()
            ax.scatter(valid_data[feat_col], valid_data[error_col], 
                      label=model_name, alpha=0.6, s=50)
    
    ax.set_xlabel(feat_name, fontsize=11, fontweight='bold')
    ax.set_ylabel('Absolute Error (pp)', fontsize=11, fontweight='bold')
    ax.set_title(f'{feat_name} vs Prediction Error', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Country Features vs Model Prediction Errors', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('error_scatter_plots.png', dpi=300, bbox_inches='tight')
plt.savefig('error_scatter_plots.pdf', bbox_inches='tight')
print("   ✓ Saved error_scatter_plots.png")
print("   ✓ Saved error_scatter_plots.pdf")
plt.close()

# Figure 3: Geographic error patterns
fig = plt.figure()
fig.set_size_inches(14, 10)
fig.set_dpi(300)
matplotlib.rcParams.update({})
axes = []
for i in range(4):
    ax = fig.add_subplot(2, 2, i + 1)
    axes.append(ax)

for idx, (model_name, col) in enumerate(models_to_analyze.items()):
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    ax = axes[idx]
    
    continent_errors = stage8_df.groupby('continent')[error_col].mean().sort_values()
    
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(continent_errors)))
    continent_errors.plot(kind='barh', ax=ax, color=colors)
    
    ax.set_xlabel('Mean Absolute Error (pp)', fontsize=11, fontweight='bold')
    ax.set_ylabel('')
    ax.set_title(f'{model_name}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Model Errors by Continent', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('error_by_continent.png', dpi=300, bbox_inches='tight')
plt.savefig('error_by_continent.pdf', bbox_inches='tight')
print("   ✓ Saved error_by_continent.png")
print("   ✓ Saved error_by_continent.pdf")
plt.close()

# Figure 4: Development level patterns
fig = plt.figure()
fig.set_size_inches(12, 6)
fig.set_dpi(300)
matplotlib.rcParams.update({})
ax = fig.add_subplot(111)

x = np.arange(4)
width = 0.2

quartile_order = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']

for idx, (model_name, col) in enumerate(models_to_analyze.items()):
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    errors = []
    for quartile in quartile_order:
        subset = stage8_df[stage8_df['gdp_quartile'] == quartile]
        if len(subset) > 0:
            errors.append(subset[error_col].mean())
        else:
            errors.append(0)
    
    offset = (idx - 1.5) * width
    ax.bar(x + offset, errors, width, label=model_name, alpha=0.8)

ax.set_xlabel('GDP Quartile', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Absolute Error (pp)', fontsize=12, fontweight='bold')
ax.set_title('Model Errors by Economic Development Level', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(quartile_order, rotation=0)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('error_by_gdp_quartile.png', dpi=300, bbox_inches='tight')
plt.savefig('error_by_gdp_quartile.pdf', bbox_inches='tight')
print("   ✓ Saved error_by_gdp_quartile.png")
print("   ✓ Saved error_by_gdp_quartile.pdf")
plt.close()

print("\n" + "="*80)
print("ERROR PATTERN ANALYSIS COMPLETE!")
print("="*80)

print("\nKey Files Created:")
print("   - error_correlations.csv")
print("   - error_feature_correlations.png / .pdf")
print("   - error_scatter_plots.png / .pdf")
print("   - error_by_continent.png / .pdf")
print("   - error_by_gdp_quartile.png / .pdf")

print("\nNext step: Run 3_case_study_analysis.py")

ERROR PATTERN ANALYSIS
Analyzing what predicts each model's prediction errors
Uses existing predictions - NO new API calls needed

1. LOADING DATA
✓ Loaded 1000 predictions
✓ Countries: 125
✓ Analyzing Stage 8: 125 countries
✓ Calculated errors for Llama
✓ Calculated errors for GPT
✓ Calculated errors for Claude
✓ Calculated errors for Gemini

2. CORRELATION ANALYSIS: Features vs Errors

Llama:
------------------------------------------------------------
   GDP per capita           : r = -0.070, p = 0.4395 
   Education (years)        : r = -0.086, p = 0.3470 
   Religious importance     : r =  0.099, p = 0.2889 
   HDI                      : r = -0.027, p = 0.7668 
   Top 1% income            : r = -0.083, p = 0.3561 
   Top 1% wealth            : r = -0.068, p = 0.4537 
   Mean age                 : r = -0.023, p = 0.7974 

GPT:
------------------------------------------------------------
   GDP per capita           : r = -0.055, p = 0.5406 
   Education (years)        : r = -0.045, 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>